In [1]:
!pip install -q langchain-core langchain-text-splitters sentence-transformers langchain-huggingface

In [2]:
from langchain_core.documents import Document

documentos = [
    Document(
        page_content="Embeddings são representações vetoriais densas de texto, onde cada palavra ou trecho vira uma lista de números.",
        metadata={
            "fonte": "arquivo_01.md",
            "pagina": 1,
            "tipo": "teoria",
            "tema": "embeddings",
            "autor": "Paula Thamyres"
        }
    ),
    Document(
        page_content="Textos semanticamente parecidos geram embeddings que ficam próximos no espaço vetorial.",
        metadata={
            "fonte": "arquivo_01.md",
            "pagina": 2,
            "tipo": "teoria",
            "tema": "embeddings",
            "autor": "Paula Thamyres"
        }
    ),
    Document(
        page_content="Chunking é o processo de dividir um documento grande em pedaços menores antes de gerar os embeddings.",
        metadata={
            "fonte": "arquivo_02.md",
            "pagina": 1,
            "tipo": "teoria",
            "tema": "chunking",
            "autor": "Paula Thamyres"
        }
    ),
    Document(
        page_content="O tamanho ideal de um chunk depende do modelo de embedding e do tipo de conteúdo que está sendo indexado.",
        metadata={
            "fonte": "arquivo_02.md",
            "pagina": 3,
            "tipo": "pratica",
            "tema": "chunking",
            "autor": "Paula Thamyres"
        }
    ),
    Document(
        page_content="RAG combina busca de informação em uma base de conhecimento com a geração de texto de um modelo de linguagem.",
        metadata={
            "fonte": "arquivo_03.md",
            "pagina": 1,
            "tipo": "teoria",
            "tema": "rag",
            "autor": "Paula Thamyres"
        }
    ),
    Document(
        page_content="Tokenização é a etapa que quebra o texto em unidades menores (tokens) antes de ele ser processado pelo modelo.",
        metadata={
            "fonte": "arquivo_04.md",
            "pagina": 1,
            "tipo": "teoria",
            "tema": "tokenizacao",
            "autor": "Paula Thamyres"
        }
    ),
]

print("Lista criada com sucesso!")

Lista criada com sucesso!


In [3]:
for i, doc in enumerate(documentos, start=1):
    print(f"--- Documento {i} ---")
    print("page_content:", doc.page_content)
    print("metadata:", doc.metadata)
    print()

--- Documento 1 ---
page_content: Embeddings são representações vetoriais densas de texto, onde cada palavra ou trecho vira uma lista de números.
metadata: {'fonte': 'arquivo_01.md', 'pagina': 1, 'tipo': 'teoria', 'tema': 'embeddings', 'autor': 'Paula Thamyres'}

--- Documento 2 ---
page_content: Textos semanticamente parecidos geram embeddings que ficam próximos no espaço vetorial.
metadata: {'fonte': 'arquivo_01.md', 'pagina': 2, 'tipo': 'teoria', 'tema': 'embeddings', 'autor': 'Paula Thamyres'}

--- Documento 3 ---
page_content: Chunking é o processo de dividir um documento grande em pedaços menores antes de gerar os embeddings.
metadata: {'fonte': 'arquivo_02.md', 'pagina': 1, 'tipo': 'teoria', 'tema': 'chunking', 'autor': 'Paula Thamyres'}

--- Documento 4 ---
page_content: O tamanho ideal de um chunk depende do modelo de embedding e do tipo de conteúdo que está sendo indexado.
metadata: {'fonte': 'arquivo_02.md', 'pagina': 3, 'tipo': 'pratica', 'tema': 'chunking', 'autor': 'Paula

In [4]:
print("Total de documentos:", len(documentos))

Total de documentos: 6


### Pergunta: que tipos de dado são aceitos dentro de metadata?


In [5]:
# Teste 1: metadata com uma lista
doc_com_lista = Document(
    page_content="Teste de metadata com lista.",
    metadata={"tags": ["rag", "embeddings", "chunking"]}
)
print("Metadata com lista:", doc_com_lista.metadata)
print("Tipo do campo 'tags':", type(doc_com_lista.metadata["tags"]))

print()

# Teste 2: metadata com um dicionário aninhado
doc_com_dict = Document(
    page_content="Teste de metadata com dicionário aninhado.",
    metadata={"info_extra": {"autor": "Paula", "ano": 2026, "revisado": True}}
)
print("Metadata com dict aninhado:", doc_com_dict.metadata)
print("Tipo do campo 'info_extra':", type(doc_com_dict.metadata["info_extra"]))

Metadata com lista: {'tags': ['rag', 'embeddings', 'chunking']}
Tipo do campo 'tags': <class 'list'>

Metadata com dict aninhado: {'info_extra': {'autor': 'Paula', 'ano': 2026, 'revisado': True}}
Tipo do campo 'info_extra': <class 'dict'>


**O que aconteceu:** o Document do LangChain-core aceita qualquer tipo de dado Python
dentro de metadata (é só um dict sem validação de schema) — string, número, booleano,
lista e até dicionário aninhado funcionam sem erro nenhum, como os prints acima mostram.

**Mas atenção:** isso funciona aqui porque ainda não colocamos os documentos dentro de
uma vector store de verdade. Quando formos indexar (Chroma, FAISS etc.) e quisermos
filtrar por metadata, a maioria dessas ferramentas só aceita filtros sobre valores
simples — str, int, float, bool. Listas e dicionários aninhados geralmente não podem
ser usados como filtro direto. Por isso, na prática, o ideal é manter o metadata
"achatado" (flat), com valores simples, e reservar listas/dicts só para campos que
você não pretende usar como filtro.

### Pergunta: o que acontece se você criar um Document sem passar metadata?


In [6]:
doc_sem_metadata = Document(page_content="Este documento não recebeu metadata nenhum.")
print("metadata:", doc_sem_metadata.metadata)
print("tipo:", type(doc_sem_metadata.metadata))

metadata: {}
tipo: <class 'dict'>


**O que aconteceu:** não dá erro nenhum. O Document tem um valor padrão para metadata,
que é um dicionário vazio ({}). Ou seja, metadata é opcional — se você não passar nada,
o LangChain preenche sozinho com {} em vez de deixar None ou quebrar o código.



## Exercício 2 — Projetando o schema de metadados

### Schema final (campos obrigatórios da atividade + 3 campos próprios)

| Campo | Descrição | Origem |
|---|---|---|
| fonte | nome do arquivo .md de origem | obrigatório |
| documento_id | identificador único do documento de origem | obrigatório |
| chunk_index | posição do chunk dentro do documento (0, 1, 2...) | obrigatório |
| estrategia | qual das 10 estratégias da Aula 04 gerou este chunk | obrigatório |
| chunk_size | configuração de tamanho usada no splitter | obrigatório |
| chunk_overlap | configuração de sobreposição usada no splitter | obrigatório |
| n_caracteres | tamanho real do chunk (calculado, não configurado) | obrigatório |
| tema | assunto principal do chunk (embeddings, chunking, RAG, tokenização...) | próprio |
| data_processamento | data em que o chunk foi gerado (formato AAAA-MM-DD) | próprio |
| hash_conteudo | hash (ex.: MD5) do texto do chunk | próprio |

**Justificativa dos campos próprios:**
- **tema** → permite responder "quais chunks falam sobre X?" sem reler o texto
  inteiro; é a base de um filtro temático na busca.
- **data_processamento** → permite responder "esse chunk foi gerado com a versão mais
  recente do pipeline ou é de uma indexação antiga?", ajudando a saber se é preciso
  reprocessar depois de uma mudança de estratégia.
- **hash_conteudo** → permite responder "esse chunk já existe na base ou é
  duplicado/mudou de conteúdo?", evitando indexar o mesmo trecho duas vezes.

In [7]:
import json
import hashlib
from datetime import date

texto_chunk = "RAG combina busca de informação em uma base de conhecimento com a geração de texto de um modelo de linguagem."

exemplo_chunk = {
    "fonte": "arquivo_03.md",
    "documento_id": "doc03",
    "chunk_index": 0,
    "estrategia": "recursive_character_splitter",
    "chunk_size": 500,
    "chunk_overlap": 50,
    "n_caracteres": len(texto_chunk),
    "tema": "rag",
    "data_processamento": str(date.today()),
    "hash_conteudo": hashlib.md5(texto_chunk.encode()).hexdigest()
}

print(json.dumps(exemplo_chunk, indent=2, ensure_ascii=False))

{
  "fonte": "arquivo_03.md",
  "documento_id": "doc03",
  "chunk_index": 0,
  "estrategia": "recursive_character_splitter",
  "chunk_size": 500,
  "chunk_overlap": 50,
  "n_caracteres": 109,
  "tema": "rag",
  "data_processamento": "2026-08-14",
  "hash_conteudo": "1054515a01071ed24c02065f381585ba"
}


### Aplicando o schema aos chunks da Aula 04

A célula abaixo tenta carregar o arquivo de chunks gerado na Aula 04 (ex.:
chunks_aula04.json). Se você fizer upload desse arquivo no Colab (ícone de pasta 📁
à esquerda → upload), ele será usado de verdade. Se o arquivo não existir, o notebook
usa um exemplo simulado só para não travar — troque pelo seu arquivo real depois.

In [8]:
import os

CAMINHO_CHUNKS_AULA04 = "chunks_aula04.json"  # ajuste para o nome/caminho real do seu arquivo

if os.path.exists(CAMINHO_CHUNKS_AULA04):
    with open(CAMINHO_CHUNKS_AULA04, "r", encoding="utf-8") as f:
        chunks_aula04 = json.load(f)
    print(f"{len(chunks_aula04)} chunks carregados de {CAMINHO_CHUNKS_AULA04}")
else:
    print("Arquivo da Aula 04 não encontrado — usando exemplo simulado para demonstrar o schema.")
    chunks_aula04 = [
        {"chunk_id": "doc01_test05_chunk001", "text": "Embeddings são vetores densos.", "metadata": {"page": 10, "section": "Introdução"}},
        {"chunk_id": "doc01_test05_chunk002", "text": "Chunking divide o texto em pedaços menores.", "metadata": {"page": 11, "section": "Chunking"}},
    ]

def montar_document_com_schema(chunk_bruto, estrategia, chunk_size, chunk_overlap, tema):
    texto = chunk_bruto["text"]
    metadata = {
        "fonte": chunk_bruto.get("metadata", {}).get("section", "desconhecida"),
        "documento_id": chunk_bruto["chunk_id"].split("_chunk")[0],
        "chunk_index": int(chunk_bruto["chunk_id"].split("chunk")[-1]),
        "estrategia": estrategia,
        "chunk_size": chunk_size,
        "chunk_overlap": chunk_overlap,
        "n_caracteres": len(texto),
        "tema": tema,
        "data_processamento": str(date.today()),
        "hash_conteudo": hashlib.md5(texto.encode()).hexdigest(),
    }
    return Document(page_content=texto, metadata=metadata)

documentos_aula04 = [
    montar_document_com_schema(c, estrategia="recursive_character_splitter", chunk_size=500, chunk_overlap=50, tema="rag")
    for c in chunks_aula04
]

for doc in documentos_aula04:
    print(doc.page_content)
    print(doc.metadata)
    print()

Arquivo da Aula 04 não encontrado — usando exemplo simulado para demonstrar o schema.
Embeddings são vetores densos.
{'fonte': 'Introdução', 'documento_id': 'doc01_test05', 'chunk_index': 1, 'estrategia': 'recursive_character_splitter', 'chunk_size': 500, 'chunk_overlap': 50, 'n_caracteres': 30, 'tema': 'rag', 'data_processamento': '2026-08-14', 'hash_conteudo': '32962f7bd0ec13b65edae4d70102268a'}

Chunking divide o texto em pedaços menores.
{'fonte': 'Chunking', 'documento_id': 'doc01_test05', 'chunk_index': 2, 'estrategia': 'recursive_character_splitter', 'chunk_size': 500, 'chunk_overlap': 50, 'n_caracteres': 43, 'tema': 'rag', 'data_processamento': '2026-08-14', 'hash_conteudo': 'de51ec362f74f1c39e57dbf60ad9c146'}



### Pergunta: qual campo você incluiria se precisasse citar a fonte na resposta final do RAG?

A combinação fonte + documento_id + chunk_index (às vezes junto de um campo pagina,
se existir). Isso permite montar uma citação do tipo "informação retirada de
arquivo_03.md, chunk 2", apontando exatamente para o trecho de onde a resposta veio —
não basta só o nome do arquivo, porque um arquivo pode ter dezenas de chunks.

### Pergunta: por que chunk_index é útil?

Porque quando a busca vetorial recupera um chunk, ele pode estar cortado no meio de
uma explicação (o corte de chunking não respeita o sentido do texto). Com chunk_index,
é possível identificar os chunks vizinhos (chunk_index - 1 e chunk_index + 1) do mesmo
documento_id e recuperá-los também, reconstruindo o contexto completo antes de gerar
a resposta final — sem isso, não haveria como saber qual pedaço vem antes ou depois.